In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

INITIAL_CAPITAL = 5_000_000
HOURS_PER_YEAR = 252 * 6.5
HORIZON = 168

In [0]:
# Load WALKFORWARD signals (production)
SIGNAL_TABLE = "pawan_quant_research_1.default.bt_walkforward_regime_signals"

signals_df = spark.table(SIGNAL_TABLE).toPandas()
signals_df["timestamp"] = pd.to_datetime(signals_df["timestamp"])

signals_df = signals_df.sort_values("timestamp").set_index("timestamp")

print("Signals loaded:", len(signals_df))

In [0]:
prices = spark.table("pawan_quant_research_1.default.bronze_bt_market_1h") \
    .filter("symbol IN ('SPY','QQQ')") \
    .toPandas()

prices["timestamp"] = pd.to_datetime(prices["timestamp"])

prices = prices.pivot_table(
    index="timestamp",
    columns="symbol",
    values="close",
    aggfunc="last"
)

df = signals_df.join(prices, how="left")

df["SPY"] = df["SPY"].ffill()
df["QQQ"] = df["QQQ"].ffill()

In [0]:
df["fwd_return_168h"] = df["SPY"].shift(-HORIZON) / df["SPY"] - 1

In [0]:
def label_regime(ret):
    if pd.isna(ret): return np.nan
    if ret > 0.005: return 0
    if ret < -0.005: return 2
    return 1

df["actual_regime"] = df["fwd_return_168h"].apply(label_regime)

In [0]:
valid = df.dropna(subset=["actual_regime"]).copy()

In [0]:
cm = confusion_matrix(valid["actual_regime"], valid["predicted_regime"])
print("Confusion Matrix:\n", cm)

In [0]:
def map_direction(regime):
    return {0:1, 1:0, 2:-1}.get(regime, 0)

valid["pred_dir"] = valid["predicted_regime"].map(map_direction)
valid["act_dir"] = np.sign(valid["fwd_return_168h"])

directional_acc = (valid["pred_dir"] == valid["act_dir"]).mean()
print("Directional Accuracy:", directional_acc)

In [0]:
valid["actual_shift"] = valid["actual_regime"].diff() != 0
valid["pred_shift"] = valid["predicted_regime"].diff() != 0

tp = (valid["actual_shift"] & valid["pred_shift"]).sum()
recall = tp / valid["actual_shift"].sum()

print("Transition Recall:", recall)

In [0]:
regime_returns = valid.groupby("predicted_regime")["fwd_return_168h"].agg(
    ["mean","std","count"]
)

regime_returns["sharpe"] = regime_returns["mean"] / regime_returns["std"]

print(regime_returns)

In [0]:
# map regimes to ordered score
score_map = {0:1, 1:0, 2:-1}

valid["score"] = valid["predicted_regime"].map(score_map)

ic = valid["score"].corr(valid["fwd_return_168h"])

print("Information Coefficient:", ic)

In [0]:
valid["hit"] = valid["pred_dir"] * valid["fwd_return_168h"] > 0

hit_ratio = valid["hit"].mean()

print("Hit Ratio:", hit_ratio)

In [0]:
capital = INITIAL_CAPITAL
portfolio = [capital]
active = "START"

for i in range(1, len(valid)):
    
    r = valid["predicted_regime"].iloc[i-1]

    if r == 0:
        ret = valid["QQQ"].iloc[i] / valid["QQQ"].iloc[i-1] - 1
        target = "QQQ"
    elif r == 1:
        ret = valid["SPY"].iloc[i] / valid["SPY"].iloc[i-1] - 1
        target = "SPY"
    else:
        ret = 0
        target = "CASH"

    cost = 0.0005 if target != active else 0
    capital *= (1 + ret - cost)

    portfolio.append(capital)
    active = target

valid = valid.iloc[:len(portfolio)].copy()
valid["portfolio"] = portfolio

In [0]:
returns = valid["portfolio"].pct_change().dropna()

years = len(valid) / HOURS_PER_YEAR

cagr = (valid["portfolio"].iloc[-1] / valid["portfolio"].iloc[0])**(1/years) - 1

vol = returns.std() * np.sqrt(HOURS_PER_YEAR)

sharpe = returns.mean() / returns.std() * np.sqrt(HOURS_PER_YEAR)

max_dd = (valid["portfolio"] / valid["portfolio"].cummax() - 1).min()

print("\nPerformance Metrics")
print("CAGR:", cagr)
print("Volatility:", vol)
print("Sharpe:", sharpe)
print("Max Drawdown:", max_dd)

In [0]:
switches = (valid["predicted_regime"].diff() != 0).sum()
turnover = switches / len(valid)

print("Turnover:", turnover)

In [0]:
plt.figure(figsize=(12,6))
plt.plot(valid.index, valid["portfolio"])
plt.title("Walkforward Strategy Equity Curve")
plt.show()

In [0]:
df_export = valid.copy()

df_export = df_export[[
    "predicted_regime",
    "fwd_return_168h"
]]

df_export = df_export.rename(columns={
    "fwd_return_168h": "actual_return_168h"
})

# map predicted → expected return (empirical)
regime_map = valid.groupby("predicted_regime")["actual_return_168h"].mean().to_dict()

df_export["predicted_return_168h"] = df_export["predicted_regime"].map(regime_map)

OUTPUT_PATH = "/Volumes/pawan_quant_research_1/default/market_data/v4_walkforward_prediction_vs_actual.csv"

df_export.to_csv(OUTPUT_PATH)

print("Saved:", OUTPUT_PATH)
print("Regime return map:", regime_map)

In [0]:
%pip install xlsxwriter

In [0]:
# ============================================================
# V4 CONFUSION MATRIX ANALYSIS (HOURLY → 168H HORIZON)
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import shutil


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

HORIZON = 168  # 1 week ahead

LABEL_MAP = {
    0: "Expansion",
    1: "Slowdown",
    2: "Contraction"
}

OUTPUT_PATH = "/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting/confusion_matrix.xlsx"
#shutil.copy("/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting", 
#            "/Volumes/pawan_quant_research_1/default/market_data/v4_confusion_matrices.xlsx")

# ------------------------------------------------------------
# LOAD WALKFORWARD SIGNALS (PRODUCTION)
# ------------------------------------------------------------

SIGNAL_TABLE = "pawan_quant_research_1.default.bt_walkforward_regime_signals"

signals_df = spark.table(SIGNAL_TABLE).toPandas()
signals_df["timestamp"] = pd.to_datetime(signals_df["timestamp"])

signals_df = signals_df.sort_values("timestamp").set_index("timestamp")

# ------------------------------------------------------------
# LOAD PRICES
# ------------------------------------------------------------

prices = spark.table("pawan_quant_research_1.default.bronze_bt_market_1h") \
    .filter("symbol IN ('SPY')") \
    .toPandas()

prices["timestamp"] = pd.to_datetime(prices["timestamp"])

prices = prices.pivot_table(
    index="timestamp",
    columns="symbol",
    values="close",
    aggfunc="last"
)

df = signals_df.join(prices, how="left")
df["SPY"] = df["SPY"].ffill()

# ------------------------------------------------------------
# CREATE FORWARD RETURNS
# ------------------------------------------------------------

df["fwd_return_168h"] = df["SPY"].shift(-HORIZON) / df["SPY"] - 1

# ------------------------------------------------------------
# CREATE ACTUAL REGIME LABELS
# ------------------------------------------------------------

def label_regime(ret):
    if pd.isna(ret): return np.nan
    if ret > 0.005: return 0
    if ret < -0.005: return 2
    return 1

df["actual_regime"] = df["fwd_return_168h"].apply(label_regime)

# ------------------------------------------------------------
# CLEAN DATA
# ------------------------------------------------------------

valid = df.dropna(subset=["actual_regime", "predicted_regime"]).copy()

# ------------------------------------------------------------
# FILTER: MARKET HOURS ONLY (US EQUITY HOURS)
# ------------------------------------------------------------

# Ensure timestamp is timezone-aware (assume UTC from Databricks)
if valid.index.tz is None:
    valid.index = valid.index.tz_localize("UTC")

# Convert to US/Eastern
valid["timestamp_est"] = valid.index.tz_convert("US/Eastern")

# Filter weekdays only (Mon–Fri)
valid = valid[valid["timestamp_est"].dt.dayofweek < 5]

# Filter market hours: 09:30–16:00 EST
valid = valid[
    (valid["timestamp_est"].dt.time >= pd.to_datetime("09:30").time()) &
    (valid["timestamp_est"].dt.time <= pd.to_datetime("16:00").time())
]

# Optional: keep only clean hourly bars (avoid 09:30 mismatch)
valid = valid[valid["timestamp_est"].dt.minute == 0]

print("After market-hours filter:", len(valid))

valid["actual_regime"] = valid["actual_regime"].astype(int)
valid["predicted_regime"] = valid["predicted_regime"].astype(int)

# Add year and month
valid["year"] = valid.index.year
valid["month"] = valid.index.to_period("M").astype(str)

# ------------------------------------------------------------
# FUNCTION: CONFUSION MATRIX (WITH LABELS)
# ------------------------------------------------------------

def compute_confusion(df_subset):

    cm = confusion_matrix(
        df_subset["actual_regime"],
        df_subset["predicted_regime"],
        labels=[0,1,2]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual Expansion","Actual Slowdown","Actual Contraction"],
        columns=["Pred Expansion","Pred Slowdown","Pred Contraction"]
    )

    return cm_df

# ============================================================
# 1. FULL PERIOD CONFUSION MATRIX
# ============================================================

cm_full = compute_confusion(valid)

print("\n=== FULL PERIOD CONFUSION MATRIX ===\n")
print(cm_full)

# ============================================================
# 2. YEAR-WISE CONFUSION MATRICES (2021–2026)
# ============================================================

year_matrices = {}

print("\n=== YEAR-WISE CONFUSION MATRICES ===\n")

for yr in sorted(valid["year"].unique()):
    
    df_year = valid[valid["year"] == yr]
    
    if len(df_year) == 0:
        continue
    
    cm_year = compute_confusion(df_year)
    year_matrices[yr] = cm_year
    
    print(f"\n--- {yr} ---\n")
    print(cm_year)

# ============================================================
# 3. MONTHLY CONFUSION MATRICES → EXCEL
# ============================================================

monthly_groups = valid.groupby("month")

with pd.ExcelWriter(OUTPUT_PATH, engine="xlsxwriter") as writer:

    # Sheet 1: Full period
    cm_full.to_excel(writer, sheet_name="Full_Period")

    # Sheet 2+: Year-wise
    for yr, cm in year_matrices.items():
        sheet_name = f"Year_{yr}"
        cm.to_excel(writer, sheet_name=sheet_name)

    # Monthly sheets
    for month, df_month in monthly_groups:

        cm_month = compute_confusion(df_month)

        # Excel sheet name limit = 31 chars
        sheet_name = month.replace("-", "_")

        cm_month.to_excel(writer, sheet_name=sheet_name)

print(f"\nExcel exported to: {OUTPUT_PATH}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(4, 3))
sns.heatmap(
    cm_full,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Full Period Confusion Matrix")
plt.tight_layout()
plt.savefig("/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting/cm_full_heatmap.png", dpi=200)
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Full-period heatmap (same as before)
plt.figure(figsize=(4, 3))
sns.heatmap(
    cm_full,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Full Period Confusion Matrix")
plt.tight_layout()
plt.savefig(
    "/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting/cm_full_heatmap.png",
    dpi=200
)
plt.show()

# Year-wise heatmaps (one figure per year)
for yr, cm_year in year_matrices.items():
    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm_year,
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"{yr} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        f"/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting/cm_year_{yr}_heatmap.png",
        dpi=200
    )
    plt.show()

In [0]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

years = sorted(year_matrices.keys())
n_years = len(years)

n_cols = min(3, n_years)          # up to 3 per row
n_rows = math.ceil(n_years / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 3 * n_rows)
)

# axes could be 1D or 2D; flatten for easy indexing
axes = axes.flatten() if n_years > 1 else [axes]

for i, yr in enumerate(years):
    ax = axes[i]
    cm_year = year_matrices[yr]
    sns.heatmap(
        cm_year,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax
    )
    ax.set_title(f"{yr}")

# Hide any unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Year-wise Confusion Matrices", y=1.02)
plt.tight_layout()
plt.savefig(
    "/Workspace/Users/pawan.jatale@mubama.group/V4_WF_RT_Regime_Strategy_Retraining_github/04_Backtesting/cm_years_all_heatmap.png",
    dpi=200
)
plt.show()